# Horn–Schunck optical flow: build, validate, benchmark and profile on Colab

Runtime → **Change runtime type → GPU** (any NVIDIA GPU, e.g. T4) before running.

This notebook clones the repository, builds it with CMake + nvcc, checks the CUDA
results against the CPU reference, runs the benchmark matrix and the profiler, and
shows the generated charts. Results land in `results/` inside the clone; download them
with the last cell.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2
!cmake --version | head -1
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread|Core'

In [ ]:
%cd /content
!rm -rf serial-parallel-computation-optical-flow-algorithm
!git clone --depth 1 https://github.com/vigildas05/serial-parallel-computation-optical-flow-algorithm.git
%cd serial-parallel-computation-optical-flow-algorithm

## Build
`CMAKE_CUDA_ARCHITECTURES=native` (the default) compiles for the GPU attached to this runtime.

In [ ]:
!apt-get -qq update && apt-get -qq install -y libopencv-dev > /dev/null
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release 2>&1 | grep -E 'CUDA|OpenMP|OpenCV|Error'
!cmake --build build -j"$(nproc)" 2>&1 | tail -3

## Correctness
CPU Jacobi variants must match the OpenMP reference bit for bit; CUDA kernels must agree within float tolerance.

In [ ]:
!./build/opt-flow-parallel validate --size 37x5,641x479,640x480,1920x1080 --iters 100,1000

## Benchmarks
The full matrix (3 sizes × 3 iteration counts × all versions, 10 runs each) takes a while on
Colab's 2-vCPU machines, mostly because of the serial baselines at 1920×1080. Reduce `RUNS` or
`ITERS` for a quicker pass; the scripts record the actual run count in the CSV.

In [ ]:
!RUNS=10 THREADS=1,2,max bash scripts/run_benchmarks.sh build/opt-flow-parallel

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open('results/tables.md').read()))
for f in ['speedup', 'thread_scaling', 'convergence']:
    display(Image(f'results/{f}.png'))

## Profiling
Nsight Compute needs GPU performance-counter access. If `ncu` reports
`ERR_NVGPUCTRPERM`, the runtime does not allow profiling and only the Nsight Systems
timeline (if `nsys` is installed) will be produced.

In [ ]:
!bash scripts/profile.sh build/opt-flow-parallel
!cat results/profile/summary.md 2>/dev/null || echo "no ncu summary"

## Flow visualizations (HSV: hue = direction, brightness = magnitude)

In [ ]:
import glob
for f in sorted(glob.glob('output/flow_hsv_*.png')):
    print(f); display(Image(f, width=480))

In [ ]:
!zip -qr results.zip results output/flow_hsv_*.png
from google.colab import files
files.download('results.zip')